# Concurrencia Avanzada en Go

Go fue diseñado con la concurrencia como pilar fundamental. Utiliza el modelo CSP (Communicating Sequential Processes), donde las goroutines se comunican a través de canales para evitar el uso de memoria compartida siempre que sea posible.

## 1. Goroutines y sync.WaitGroup

Aunque `go func()` lanza una goroutine, necesitamos una forma de esperar a que terminen antes de que el programa principal finalice.

In [ ]:
import (
    "fmt"
    "sync"
    "time"
)

func trabajador(id int, wg *sync.WaitGroup) {
    defer wg.Done() // Decrementa el contador al terminar
    
    fmt.Printf("Trabajador %d iniciando\n", id)
    time.Sleep(time.Second)
    fmt.Printf("Trabajador %d finalizado\n", id)
}

func main() {
    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        wg.Add(1) // Incrementa el contador
        go trabajador(i, &wg)
    }

    wg.Wait() // Bloquea hasta que el contador sea 0
    fmt.Println("Todos los trabajadores terminaron")
}

## 2. Canales (Channels) con Buffer y Select

- **Unbuffered**: El envío bloquea hasta que alguien recibe.
- **Buffered**: El envío solo bloquea cuando el buffer está lleno.
- **Select**: Permite esperar en múltiples operaciones de canal.

In [ ]:
ch1 := make(chan string)
ch2 := make(chan string)

go func() {
    time.Sleep(1 * time.Second)
    ch1 <- "uno"
}()
go func() {
    time.Sleep(2 * time.Second)
    ch2 <- "dos"
}()

for i := 0; i < 2; i++ {
    select {
    case msg1 := <-ch1:
        fmt.Println("Recibido:", msg1)
    case msg2 := <-ch2:
        fmt.Println("Recibido:", msg2)
    case <-time.After(3 * time.Second):
        fmt.Println("Timeout")
    }
}

## 3. El Paquete Context

En Go, `context.Context` es vital para manejar la cancelación, los tiempos de espera y el paso de metadatos a través de las fronteras de la API y los límites de los procesos.

In [ ]:
import "context"

func operacionLenta(ctx context.Context) {
    select {
    case <-time.After(2 * time.Second):
        fmt.Println("Operación finalizada")
    case <-ctx.Done(): // Cancelado por el padre
        fmt.Println("Operación abortada:", ctx.Err())
    }
}

func main() {
    // Contexto con timeout de 1 segundo
    ctx, cancel := context.WithTimeout(context.Background(), 1*time.Second)
    defer cancel()

    go operacionLenta(ctx)

    <-ctx.Done()
}

## 4. Sincronización Avanzada: Mutex y Once

Cuando los canales no son suficientes y necesitamos proteger el estado compartido.

In [ ]:
type Contador struct {
    mu    sync.Mutex
    valor int
}

func (c *Contador) Inc() {
    c.mu.Lock()         // Acceso exclusivo
    defer c.mu.Unlock() // Liberar al salir
    c.valor++
}

// sync.Once garantiza que una función se ejecute exactamente una vez
var once sync.Once
inicializar := func() {
    fmt.Println("Solo me verás una vez")
}

for i := 0; i < 10; i++ {
    go func() {
        once.Do(inicializar)
    }()
}

## 5. Patrón Worker Pool

Un patrón común para limitar el número de goroutines concurrentes procesando una cola de trabajo.

In [ ]:
func worker(id int, trabajos <-chan int, resultados chan<- int) {
    for j := range trabajos {
        fmt.Printf("Worker %d procesando trabajo %d\n", id, j)
        time.Sleep(time.Second)
        resultados <- j * 2
    }
}

func main() {
    const numTrabajos = 5
    trabajos := make(chan int, numTrabajos)
    resultados := make(chan int, numTrabajos)

    // Lanzar 3 workers
    for w := 1; w <= 3; w++ {
        go worker(w, trabajos, resultados)
    }

    // Enviar trabajos
    for j := 1; j <= numTrabajos; j++ {
        trabajos <- j
    }
    close(trabajos)

    // Recoger resultados
    for a := 1; a <= numTrabajos; a++ {
        <-resultados
    }
}

## 6. Race Detector

Go incluye una herramienta potente para detectar condiciones de carrera en tiempo de ejecución. 
Se utiliza al ejecutar tests o programas:

```bash
go run -race programa.go
go test -race ./...
```